# 02 — Event Definition: Forward-Looking Drawdown Labels

**Thesis Chapter 1 — Probabilistic Risk Estimation in Financial Markets using GRU Neural Networks**

## Purpose

This notebook constructs the **binary classification target** `y(t)` that the GRU model
will learn to predict. It answers the question:

> *"Given today's closing price, will the asset lose at least X% within the next H trading days?"*

## Methodology

**Forward-looking drawdown (current-to-future-min):**

$$DD(t) = \frac{\min_{u \in [t+1,\, t+H]} P_u \;-\; P_t}{P_t}$$

- Negative values indicate a price drop from today.
- Binary label: `y(t) = 1` if `DD(t) <= -X%`, else `y(t) = 0`.

## Causality & No Lookahead Bias

**Critical:** The label `y(t)` uses *future* prices `P_{t+1} ... P_{t+H}` — this is
intentional because it is the TARGET we want to PREDICT, not a feature. At inference
time the model sees only past data. The label is never used as an input feature.

The last H rows of the dataset will have `NaN` labels because the full forward window
is unavailable. These rows are excluded from training and evaluation.

## Pipeline Position

```
01_data_pipeline  →  [02_event_definition]  →  03_feature_engineering  →  04_model
   (OHLCV)              (adds labels)              (adds features)         (GRU)
```


## 0. Environment Setup & Configuration

Load libraries and set event-definition parameters:
- `horizon_days` (H): how many trading days to look ahead
- `drawdown_threshold` (X): minimum loss magnitude to flag as an "event"
- `drawdown_method`: which drawdown formula to use

These parameters directly control the class balance and the economic meaning of the label.


In [ ]:
# ─── Environment Setup ─────────────────────────────────────────────────────────
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# ─── Project Paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "figures"

# ─── Event Definition Configuration ───────────────────────────────────────────
# These parameters control the meaning of a "risk event":
#   - horizon_days: forward window length (H)
#   - drawdown_threshold: loss magnitude that triggers label=1
#   - drawdown_method: mathematical definition of drawdown
CFG = {
    "primary_symbol": "^NDX",                      # Asset to label
    "horizon_days": 3,                              # H = look-ahead window in trading days
    "drawdown_threshold": 0.03,                     # X = 3% loss threshold
    "drawdown_method": "current_to_future_min",     # Primary method for thesis
    # ── Method descriptions ──
    # "current_to_future_min": DD(t) = [min future price - current] / current
    #   → Answers: "What is the worst-case % loss from TODAY's price in next H days?"
    # "peak_to_trough": DD(t) = (peak - trough) / peak within [t+1, t+H]
    #   → Answers: "What is the max intra-window drawdown in next H days?"
}

print("Setup complete")
print(f"Primary symbol: {CFG['primary_symbol']}")
print(f"Horizon days: {CFG['horizon_days']}")
print(f"Drawdown threshold: {CFG['drawdown_threshold']*100}%")
print(f"Drawdown method: {CFG.get('drawdown_method', 'current_to_future_min')}")
print(f"Data directory: {DATA_DIR}")


## Load Clean Dataset from Pipeline Step 01

We load the canonical OHLCV dataset produced by `01_data_pipeline.ipynb`.
The file is expected at `data/clean_{SYMBOL}.parquet` (or `.pkl` fallback).

If the file does not exist, the pipeline dependency is broken and we raise an
explicit error directing the user to run notebook 01 first.


In [ ]:
# ─── Load Clean Dataset ────────────────────────────────────────────────────────
symbol = CFG["primary_symbol"]

# Try parquet first (preferred), fall back to pickle
clean_file = DATA_DIR / f"clean_{symbol}.parquet"
if not clean_file.exists():
    clean_file = DATA_DIR / f"clean_{symbol}.pkl"

if not clean_file.exists():
    raise FileNotFoundError(
        f"Clean dataset not found: {clean_file}. Run 01_data_pipeline.ipynb first."
    )

print(f"Loading clean dataset from: {clean_file}")

if clean_file.suffix == '.parquet':
    df = pd.read_parquet(clean_file)
    # Restore DatetimeIndex (parquet saves index as a regular column)
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df = df.set_index(col)
            break
else:
    df = pd.read_pickle(clean_file)

print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nFirst few rows:")
df.head()


## 1. Compute Forward-Looking Drawdown Over Horizon H

### Mathematical Definition (current-to-future-min)

For each trading day $t$:

$$DD(t) = \frac{\min\{P_{t+1}, P_{t+2}, \ldots, P_{t+H}\} - P_t}{P_t}$$

**Interpretation:** the worst-case percentage loss an investor experiences if they
buy at today's close and hold for up to H days.

### Implementation Notes

- We use a simple loop rather than `rolling().min()` because pandas' rolling window
  is backward-looking by default. The forward window `[t+1, t+H]` is more naturally
  expressed with explicit indexing.
- The last H rows get `NaN` because we cannot see H days into the future from them.
- **No lookahead bias:** This is the TARGET variable, not a feature. The model never
  sees drawdown or label values as input — only historical OHLCV and derived features.


In [ ]:
# ─── Forward Drawdown Computation ─────────────────────────────────────────────
H = CFG["horizon_days"]  # Number of days to look ahead
drawdown_method = CFG.get("drawdown_method", "current_to_future_min")

if drawdown_method == "current_to_future_min":
    # ═══════════════════════════════════════════════════════════════════════════
    # METHOD 1: Current-to-Future-Min (DEFAULT for thesis)
    # ═══════════════════════════════════════════════════════════════════════════
    # DD(t) = [min(P_{t+1},...,P_{t+H}) - P_t] / P_t
    # Result is NEGATIVE when price falls (worst case = most negative)
    # This directly answers: "What's the max I could lose from buying today?"
    
    forward_min_close = pd.Series(index=df.index, dtype=float)
    
    for i in range(len(df)):
        if i + H < len(df):
            # Window: [t+1, t+H] — strictly FUTURE days (no peeking at today)
            forward_min_close.iloc[i] = df['Close'].iloc[i+1:i+H+1].min()
        else:
            # Insufficient future data → mark as NaN (cannot compute label)
            forward_min_close.iloc[i] = np.nan
    
    # Drawdown: fractional change from current price to worst future price
    # Negative means loss, e.g., -0.05 = 5% drop from current price
    drawdown = (forward_min_close - df['Close']) / df['Close']
    
    print(f"Computed forward drawdown (current-to-future-min) over {H}-day horizon")
    print("Definition: DD(t) = min{{u in [t+1, t+H]}} (P_u - P_t) / P_t")
    print(f"Interpretation: Worst-case % loss from current price in next {H} days")
    
elif drawdown_method == "peak_to_trough":
    # ═══════════════════════════════════════════════════════════════════════════
    # METHOD 2: Peak-to-Trough (alternative, measures intra-window volatility)
    # ═══════════════════════════════════════════════════════════════════════════
    # DD(t) = (max - min) / max within [t+1, t+H]
    # Always non-negative; measures range of price movement in forward window
    
    forward_max_close = pd.Series(index=df.index, dtype=float)
    forward_min_close = pd.Series(index=df.index, dtype=float)
    
    for i in range(len(df)):
        if i + H < len(df):
            window = df['Close'].iloc[i+1:i+H+1]
            forward_max_close.iloc[i] = window.max()
            forward_min_close.iloc[i] = window.min()
        else:
            forward_max_close.iloc[i] = np.nan
            forward_min_close.iloc[i] = np.nan
    
    drawdown = (forward_max_close - forward_min_close) / forward_max_close
    
    print(f"Computed forward drawdown (peak-to-trough) over {H}-day horizon")
    print("Definition: DD(t) = (peak - trough) / peak within [t+1, t+H]")
    
else:
    raise ValueError(
        f"Unknown drawdown_method: {drawdown_method}. "
        "Must be 'current_to_future_min' or 'peak_to_trough'"
    )

# ─── Report ───────────────────────────────────────────────────────────────────
print(f"\nDrawdown statistics (excluding NaN):")
print(drawdown.describe())
print(f"\nNote: Last {H} rows set to NaN (incomplete forward window)")
print(f"\nSample drawdown values (first 10):")
print(drawdown.head(10))


## 2. Create Binary Event Label

Convert the continuous drawdown into a binary classification target:

$$y(t) = \begin{cases} 1 & \text{if } DD(t) \leq -X\% \quad (\text{"risk event"}) \\ 0 & \text{otherwise} \end{cases}$$

**Threshold choice (X):** This is a modelling decision that trades off:
- **Smaller X** (e.g., 2%): more events → better class balance, but many are trivial noise
- **Larger X** (e.g., 10%): fewer events → severe class imbalance, but each event is economically meaningful

The threshold is configurable in `CFG` and should be tuned as part of the experimental design.

**NaN handling:** Rows where drawdown is NaN (last H days) get NaN labels —
they will be excluded from all train/val/test splits.


In [ ]:
# ─── Binary Label Construction ────────────────────────────────────────────────
X = CFG["drawdown_threshold"]   # e.g., 0.03 = 3%
threshold = -X                   # Negative because drawdown is negative for losses

# Label = 1 if drawdown is at or below the threshold (i.e., loss >= X%)
# Label = 0 if drawdown is above the threshold (i.e., loss < X% or price rose)
label = (drawdown <= threshold).astype(int)

# Propagate NaN from drawdown to label (last H rows have no valid forward window)
label = label.where(~drawdown.isna(), np.nan)

# ─── Attach to DataFrame ──────────────────────────────────────────────────────
df['drawdown'] = drawdown
df['label'] = label

# ─── Report ───────────────────────────────────────────────────────────────────
print(f"Created binary label with threshold: {threshold*100}%")
print(f"  Label = 1 if drawdown <= {threshold*100}% (loss of {X*100}% or more)")
print(f"  Label = 0 otherwise")
print(f"\nLabel statistics:")
print(f"  Total valid labels: {label.notna().sum()}")
print(f"  NaN labels (last {H} days): {label.isna().sum()}")
print(f"  Class 0 count: {(label == 0).sum()}")
print(f"  Class 1 count: {(label == 1).sum()}")
print(f"\nSample rows with label=1 (events):")
print(df[df['label'] == 1][['Close', 'drawdown', 'label']].head(10))


## 3. Label Distribution & Class Balance Analysis

Understanding the class balance is crucial for model design:
- Severe imbalance (e.g., <5% events) requires specialised techniques:
  class weights, focal loss, oversampling (SMOTE), or threshold calibration.
- The event rate should be economically plausible: NASDAQ-100 drawdowns of 3%+
  in a 3-day window should occur during market stress but not daily.

We also examine drawdown statistics conditioned on each class to verify
the threshold cleanly separates "normal" days from "stress" days.


In [265]:
# Label distribution
print("=" * 60)
print("LABEL DISTRIBUTION")
print("=" * 60)

valid_labels = label.dropna()
class_counts = valid_labels.value_counts().sort_index()
class_pct = (class_counts / len(valid_labels) * 100).round(2)

print(f"\nTotal valid labels: {len(valid_labels)}")
print(f"NaN labels (last {H-1} days): {label.isna().sum()}")
print(f"\nClass distribution:")
print(f"  Class 0 (no event): {class_counts.get(0, 0)} ({class_pct.get(0, 0)}%)")
print(f"  Class 1 (event):    {class_counts.get(1, 0)} ({class_pct.get(1, 0)}%)")
print(f"\nClass balance ratio (0:1): {class_counts.get(0, 0)}:{class_counts.get(1, 0)}")

# Summary statistics for drawdown
print(f"\nDrawdown statistics (valid data only):")
print(df['drawdown'].describe())

# Drawdown statistics by label
print(f"\nDrawdown statistics by label:")
print(df.groupby('label')['drawdown'].describe())


LABEL DISTRIBUTION

Total valid labels: 5372
NaN labels (last 2 days): 3

Class distribution:
  Class 0 (no event): 4532 (84.36%)
  Class 1 (event):    840 (15.64%)

Class balance ratio (0:1): 4532:840

Drawdown statistics (valid data only):
count    5372.000000
mean       -0.010195
std         0.025317
min        -0.274500
25%        -0.020676
50%        -0.006616
75%         0.004349
max         0.157325
Name: drawdown, dtype: float64

Drawdown statistics by label:
        count      mean       std       min       25%       50%       75%  \
label                                                                       
0.0    4532.0 -0.002440  0.015247 -0.029989 -0.012835 -0.003281  0.006345   
1.0     840.0 -0.052037  0.027749 -0.274500 -0.058235 -0.043099 -0.035484   

            max  
label            
0.0    0.157325  
1.0   -0.030010  


## Visualisation: Events on Price Chart, Drawdown Distribution, Rolling Event Rate

Three complementary views:

1. **Events on Price Chart** — red markers show when label=1 occurs; clusters should
   align with known crisis periods (dot-com, GFC, COVID, 2022 rate hikes).
2. **Drawdown Distribution** — histogram of DD(t); the vertical red line shows the
   threshold. Events are the left tail beyond this line.
3. **Rolling Event Rate** — 252-day moving average of `label` shows how the regime
   changes over time. High event rate = bear market / elevated volatility.


In [ ]:
# ─── Plot Utilities ────────────────────────────────────────────────────────────
def save_fig(name: str, dpi: int = 150):
    """Save current matplotlib figure to figures/ directory."""
    if not name.lower().endswith(".png"):
        name = f"{name}.png"
    out = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(out, dpi=dpi, bbox_inches="tight")
    return out

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

# ─── Plot 1: Close Price with Event Markers ───────────────────────────────────
# Events (label=1) are shown as red X markers overlaid on the price series.
# Visual sanity check: events should cluster during market downturns.
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['Close'], linewidth=1, alpha=0.7, label='Close Price')

# Overlay event days
events = df[df['label'] == 1]
plt.scatter(events.index, events['Close'], color='red', marker='x', s=50, 
            label=f'Events (drawdown <= {threshold*100}%)', zorder=5)
plt.title(f"{symbol} - Close Price with Event Markers")
plt.xlabel("Date")
plt.ylabel("Close Price ($)")
plt.legend()
plt.grid(True, alpha=0.3)
fig_path = save_fig(f"02_{symbol}_events_on_price")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ─── Plot 2: Drawdown Distribution ────────────────────────────────────────────
# Histogram of all DD(t) values. The threshold line separates class 0 from class 1.
# Left tail beyond the red line = "events" the model must predict.
plt.figure(figsize=(12, 5))
valid_drawdown = df['drawdown'].dropna()
plt.hist(valid_drawdown, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(x=threshold, color='red', linestyle='--', linewidth=2, 
            label=f'Threshold: {threshold*100}%')
plt.axvline(x=0, color='black', linestyle='-', linewidth=1, alpha=0.5)  # Zero reference
plt.xlabel("Drawdown")
plt.ylabel("Frequency")
plt.title(f"{symbol} - Drawdown Distribution (H={H} days)")
plt.legend()
plt.grid(True, alpha=0.3)
fig_path = save_fig(f"02_{symbol}_drawdown_distribution")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ─── Plot 3: Rolling Event Rate Over Time ────────────────────────────────────
# A 252-day (1-year) rolling mean of the binary label shows how the
# "risk regime" evolves. High values = sustained stress (bear market).
# This is useful for identifying concept drift in the label distribution.
rolling_window = 252  # ~1 trading year
event_rate = df['label'].rolling(window=rolling_window, min_periods=1).mean()

plt.figure(figsize=(14, 6))
plt.plot(df.index, event_rate * 100, linewidth=2, 
         label=f'{rolling_window}-day rolling event rate')
plt.axhline(y=class_pct.get(1, 0), color='red', linestyle='--', linewidth=1, 
           label=f'Overall event rate: {class_pct.get(1, 0)}%')
plt.xlabel("Date")
plt.ylabel("Event Rate (%)")
plt.title(f"{symbol} - Rolling Event Rate Over Time (H={H} days, threshold={threshold*100}%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, max(event_rate.max() * 100 * 1.1, 10))
fig_path = save_fig(f"02_{symbol}_event_rate_over_time")
plt.show()
print(f"Saved: {fig_path}")

# Summary statistics for the rolling event rate
print(f"\nEvent rate over time summary:")
print(f"  Overall event rate: {class_pct.get(1, 0)}%")
print(f"  Rolling {rolling_window}-day event rate:")
print(f"    Mean: {event_rate.mean()*100:.2f}%")
print(f"    Std:  {event_rate.std()*100:.2f}%")
print(f"    Min:  {event_rate.min()*100:.2f}%")
print(f"    Max:  {event_rate.max()*100:.2f}%")


## 4. Save Labeled Dataset

Persist the DataFrame (OHLCV + `log_close` + `log_ret` + `drawdown` + `label`) to disk.
This becomes the input for `03_feature_engineering.ipynb`.

**Metadata stored in `.attrs`:** horizon, threshold, event counts — enables downstream
notebooks to verify they are using the expected label definition without re-reading config.

**File naming:** `labeled_{SYMBOL}.parquet` distinguishes this from the unlabeled
`clean_{SYMBOL}.parquet` produced by notebook 01.


In [269]:
# Add metadata to dataframe attributes
df.attrs['ticker'] = symbol
df.attrs['horizon_days'] = H
df.attrs['drawdown_threshold'] = X
df.attrs['label_creation_timestamp'] = datetime.now().isoformat()
df.attrs['n_rows'] = len(df)
df.attrs['n_valid_labels'] = int(label.notna().sum())
df.attrs['n_events'] = int((label == 1).sum())
df.attrs['event_rate'] = float(class_pct.get(1, 0))

# Save labeled dataset
# Use same pattern as 01_data_pipeline (fastparquet or pickle fallback)
output_path = DATA_DIR / f"labeled_{symbol}.parquet"
df_to_save = df.reset_index()  # Convert index to column

# Try fastparquet engine (doesn't have pyarrow initialization issues)
saved = False
try:
    df_to_save.to_parquet(output_path, index=False, engine='fastparquet')
    print("Saved using fastparquet engine")
    saved = True
except (ImportError, Exception) as e:
    print(f"fastparquet not available: {e}")

# If parquet save failed, use pickle as reliable fallback
if not saved:
    print("\nNote: Parquet save unavailable due to pyarrow compatibility issues.")
    print("Using pickle format instead (reliable and preserves all data types)")
    output_path = DATA_DIR / f"labeled_{symbol}.pkl"
    df.to_pickle(output_path)
    print(f"Saved to pickle format: {output_path}")

print(f"\nLabeled dataset saved to: {output_path}")
print(f"  Rows: {len(df)}")
print(f"  Columns: {list(df.columns)}")
print(f"  Valid labels: {int(label.notna().sum())}")
print(f"  Events (label=1): {int((label == 1).sum())}")
print(f"  Event rate: {class_pct.get(1, 0)}%")
if output_path.exists():
    print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

# Verify the saved file can be loaded
if output_path.suffix == '.parquet':
    try:
        df_loaded = pd.read_parquet(output_path, engine='fastparquet')
        # Set date column back as index when loading
        for col in df_loaded.columns:
            if pd.api.types.is_datetime64_any_dtype(df_loaded[col]):
                df_loaded = df_loaded.set_index(col)
                break
        print(f"\n✓ Verification: Loaded {len(df_loaded)} rows from saved file")
    except Exception as e:
        print(f"Warning: Could not verify parquet file: {e}")
else:
    df_loaded = pd.read_pickle(output_path)
    print(f"\n✓ Verification: Loaded {len(df_loaded)} rows from saved file")


Saved using fastparquet engine

Labeled dataset saved to: C:\Users\junio\Documents\TU821-4\FINAL YEAR PROJECT\data\labeled_VDE.parquet
  Rows: 5375
  Columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'log_close', 'log_ret', 'drawdown', 'label']
  Valid labels: 5372
  Events (label=1): 840
  Event rate: 15.64%
  File size: 0.34 MB

✓ Verification: Loaded 5375 rows from saved file


## Final Summary

Print a consolidated summary of the event definition step, confirming:
- Label parameters (H, threshold)
- Class distribution
- Output file location


In [270]:
# Final summary
print("\n" + "=" * 60)
print("✓ EVENT DEFINITION COMPLETE")
print("=" * 60)
print(f"Labeled dataset saved to: {output_path}")
print(f"Plots saved to: {FIG_DIR}")
print(f"\nLabel summary:")
print(f"  Horizon: {H} days")
print(f"  Threshold: {threshold*100}%")
print(f"  Total valid labels: {int(label.notna().sum())}")
print(f"  Class 0 (no event): {class_counts.get(0, 0)} ({class_pct.get(0, 0)}%)")
print(f"  Class 1 (event):    {class_counts.get(1, 0)} ({class_pct.get(1, 0)}%)")
print(f"  NaN labels (last {H} days): {int(label.isna().sum())}")



✓ EVENT DEFINITION COMPLETE
Labeled dataset saved to: C:\Users\junio\Documents\TU821-4\FINAL YEAR PROJECT\data\labeled_VDE.parquet
Plots saved to: C:\Users\junio\Documents\TU821-4\FINAL YEAR PROJECT\figures

Label summary:
  Horizon: 3 days
  Threshold: -3.0%
  Total valid labels: 5372
  Class 0 (no event): 4532 (84.36%)
  Class 1 (event):    840 (15.64%)
  NaN labels (last 3 days): 3
